# 01. Introduction & Philosophy

## The Vision: Unstoppable Access

ConfigStream represents a paradigm shift in how we approach internet freedom tools. It was born from a simple, unyielding necessity: **access to the free and open internet is a fundamental human right**, and it should not be complicated, expensive, or fragile.

In a world of increasing digital borders, state-sponsored censorship, and fragmented networks, the "standard" solutions often fail:
*   **Commercial VPNs**: Easy to block, require payment (identity trail), and rely on central trust.
*   **Self-Hosted Servers**: Require technical expertise, credit cards, and maintenance.
*   **Public Proxies**: Historically unreliable, ephemeral, dangerous, and often broken.

**ConfigStream solves this by treating proxy aggregation not as a "collection" task, but as a high-velocity data engineering problem.**

We act as a global refinery for the open internet:
1.  **Ingest**: We consume thousands of potential access points from hundreds of chaotic sources (GitHub repositories, Telegram channels, Pastebins, APIs).
2.  **Refine**: We parse, normalize, and structurally validate these configurations, discarding the malformed and the dangerous.
3.  **Verify**: We test connectivity, latency, and protocol handshake validity in real-time using a hybrid Python/Go engine.
4.  **Secure**: We actively probe for honeypots, strip tracking metadata, and "wash" dirty IPs using Cloudflare WARP.
5.  **Enrich**: We add geolocation, ISP data, and historical reliability scores.
6.  **Distribute**: We publish the "refined fuel" (clean, working proxies) via high-availability, censorship-resistant CDNs (GitHub Pages, Cloudflare, IPFS, Hugging Face).

## The "Zero Budget" Manifesto

A core constraint of ConfigStream is to run **entirely on free-tier infrastructure**. This is not just about saving money; it is a design philosophy that ensures resilience and sustainability.

**If it costs money, it can be cancelled.**
**If it relies on a credit card, it can be traced.**

By strictly adhering to "Zero Budget," ConfigStream becomes:
*   **Uncancellable**: As long as GitHub and free CI/CD providers exist, ConfigStream exists.
*   **Sustainable**: No server bills mean the project can run indefinitely without donations or commercial models.
*   **Scalable**: We leverage the massive, distributed compute power of public CI runners rather than a single limited VPS.

### The Stack

| Component | Solution | Role |
| :--- | :--- | :--- |
| **Compute** | **GitHub Actions** (Linux Runners) | The engine. We use parallel matrix jobs to get 20+ vCPUs of concurrent processing power for free. |
| **Storage** | **GitHub Repository** | Source code and configuration. |
| **Persistence** | **GitHub Artifacts & Cache** | Ephemeral state passing. Databases (`sqlite`) are cached between runs. |
| **Hosting** | **GitHub Pages** | Static file hosting for subscriptions and the frontend. |
| **CDN** | **Fastly / Cloudflare** | GitHub Pages uses Fastly; we also deploy mirrors to Cloudflare Pages. |
| **Database** | **SQLite (Serverless)** | We treat the DB as a file. It is downloaded at the start of a job and uploaded at the end. |
| **Intelligence** | **VirusTotal (Free API)** | Passive IP reputation scanning. |
| **Edge Logic** | **Cloudflare Workers** | Stateless API endpoints and Telegram bot hosting. |
| **Mirrors** | **Hugging Face / IPFS** | Immutable, redundant storage for outputs. |

## Digital Sovereignty & The User

ConfigStream empowers the user to be their own ISP. By aggregating thousands of scattered, weak signals (individual proxies), we create a strong, unified signal (a robust network).

*   **No Login**: We never ask for email or credentials.
*   **No Logs**: We run on ephemeral runners that are wiped after execution. We cannot log user activity even if we wanted to.
*   **Client Control**: We do not force a specific client app. We output standard formats (Clash, Sing-box) that work with open-source tools users already trust.

## Design Principles

### 1. Fail Fast, Recover Faster
The internet is messy. Sources die. Proxies rot. Our pipeline is designed to be ruthless.
*   **Adaptive Timeouts**: We learn the speed of a source. If it usually responds in 1s but takes 5s today, we cut it.
*   **Circuit Breakers**: If a source fails 3 times in a row, we stop checking it for the rest of the run to save resources.

### 2. Safety First (The "No Abuse" Pledge)
We are guests on the internet infrastructure. We must behave responsibly.
*   **Passive Only**: We never port scan random ranges. We only verify IPs that have been publicly shared.
*   **Rate Limiting**: We respect source server limits.
*   **Blocklists**: We integrate FireHol Level 1 to strictly block known malicious IPs (botnets, spam sources) from ever reaching the user.

### 3. Client Agnostic
We do not believe in "one app to rule them all." Users have preferences.
*   **Native Configs**: We output tailored configurations for **Clash**, **Sing-box**, **Surge**, **Loon**, **Quantumult X**.
*   **Universal Formats**: We provide **SIP008** and **Base64** subscriptions for maximum compatibility.
*   **Field Mapping**: We handle complex mapping of protocol fields (e.g., converting VLESS flow settings to Clash's specific format).

### 4. Transparency & Verifiability
Trust is earned.
*   **Open Source**: Every line of code is visible.
*   **Traceability**: Every proxy in the final list includes metadata about its origin (`_source` field).
*   **Reproducibility**: Anyone can fork the repo and run their own private instance of ConfigStream.

### 5. Decentralization
A central point of failure is a censorship target.
*   **Multiple Mirrors**: If GitHub is blocked, we have mirrors on GitLab, Hugging Face, and IPFS.
*   **Distributed Testing**: Our WASM client allows users to verify proxies from *their* network perspective, not just ours.

## Evolution of ConfigStream

*   **v1.0 (The Script)**: A simple Python script running sequentially. Took 4 hours to process 1000 proxies.
*   **v1.1 (Async)**: Introduced `asyncio`. Reduced time to 30 minutes.
*   **v1.2 (The Pipeline)**: Introduced GitHub Actions Matrix Strategy. Distributed processing across 6 machines.
*   **v1.3 (The Hybrid)**: Added the Go sidecar (`sing-box` integration) for robust protocol testing.
*   **v2.0 (The Platform - Current)**:
    *   **WASM Edge Testing**: Client-side verification.
    *   **Smart Intelligence**: Source quality tracking and anomaly detection.
    *   **Proxy Washing**: Cleaning dirty IPs.
    *   **Vector Search**: Natural language filtering.
    *   **Full Client Support**: Surge, Loon, QX, SIP008.

ConfigStream is not just a tool; it is a demonstration of how powerful software can be built without a budget, relying on architectural ingenuity rather than capital.




---



# 02. Architecture & Design

## The "Resilient Core" Architecture

ConfigStream operates on a unique architecture we call the **"Resilient Core"**. It is designed to function in ephemeral, resource-constrained environments (like CI runners) while maintaining the statefulness and intelligence of a persistent server.

### System Overview

The system consists of three main layers:
1.  **The Control Plane (Python)**: Orchestration, business logic, parsing, and data management.
2.  **The Data Plane (Go/Sing-box)**: High-performance raw socket operations, protocol handshakes, and encryption.
3.  **The Edge Plane (WASM/JS)**: Client-side visualization, verification, and filtering.

```mermaid
graph TD
    subgraph "CI Environment (GitHub Actions)"
        A[Sources (Text/API)] -->|Fetch| B(Fetcher Module)
        B -->|Raw Content| C(Parser Module)
        C -->|Proxy Objects| D{Deduplication}
        D -->|Unique Proxies| E(Validation Layer)
        E -->|Safe Configs| F[Hybrid Tester Engine]

        subgraph "Hybrid Tester Engine"
            F -->|Complex Protocols| G[Sing-box Tester]
            F -->|Raw Socket/Perf| H[Go Batch Tester]
        end

        G -->|Results| I[Result Aggregator]
        H -->|Results| I

        I -->|Working Proxies| J(Intelligence Layer)
        subgraph "Intelligence Layer"
            J --> K[GeoIP Resolver]
            J --> L[Source Quality Tracker]
            J --> M[Anomaly Detector]
        end

        K --> N[Output Generator]
    end

    N -->|Artifacts| O[GitHub Pages / Mirrors]

    subgraph "Client Side (Browser)"
        O --> P[Frontend PWA]
        P -->|Verify| Q[WASM Tester]
    end
```

## 1. The Control Plane (Python)

Implemented in `src/configstream/`, this layer is the "brain."

*   **Pipeline Orchestration**: `pipeline.py` and `pipeline_stages.py` manage the flow of data. We use an **Async Generator** pattern (`source_producer` -> `Queue` -> `processing_consumer`) to stream data through the system, keeping memory usage constant regardless of input size.
*   **State Management via Cache**:
    *   Since the runner is wiped after every job, we persist state (reliability scores, history) in SQLite databases (`data/*.db`).
    *   These files are hashed and stored in GitHub Actions Cache (`actions/cache`).
    *   On startup, the pipeline tries to restore the cache. If successful, it loads the "memory" of previous runs.
*   **Concurrency Manager**: `concurrency_manager.py` dynamically adjusts the number of parallel tasks based on CPU load. If the runner is struggling, it backs off to prevent an OOM (Out Of Memory) kill.

## 2. The Data Plane (The Hybrid Engine)

To test 10,000+ proxies in minutes on a free runner, Python's `asyncio` loop is not enough—the GIL (Global Interpreter Lock) becomes a bottleneck. We solve this with a hybrid approach.

### The Go Sidecar (`src/go/tester`)
*   **Role**: Mass connectivity testing.
*   **Mechanism**: A compiled Go binary. Python spawns it as a subprocess and communicates via standard I/O (JSON streaming).
*   **Performance**: Go spawns thousands of lightweight Goroutines, allowing us to saturate the network interface without CPU blocking.
*   **Features**:
    *   **TCP/UDP Checks**: Fast socket opening.
    *   **TLS Handshake**: Verifies the certificate validity.
    *   **uTLS Integration**: Randomized Client Hello fingerprinting to bypass anti-bot protections on proxy servers.

### Sing-box Integration (`singbox2proxy`)
*   **Role**: Testing complex, modern protocols (Hysteria 2, Tuic, VLESS-Reality).
*   **Mechanism**: Since these protocols require complex client-side state machines, we wrap the official `sing-box` core.
*   **Process**:
    1.  Python generates a temporary, minimal `config.json` for `sing-box`.
    2.  Spawns `sing-box` listening on a random local port.
    3.  Python sends a request through that local port to a test URL (`http://cp.cloudflare.com/generate_204`).
    4.  If it returns 204, the proxy works.

## 3. The Edge Plane (WASM)

To decentralize testing and provide users with truth from *their* perspective, we moved testing to the browser.

*   **WebAssembly (WASM)**: We compile the Go tester code to WASM (`tester.wasm`).
*   **Limitations & Solutions**:
    *   Browsers cannot open raw TCP sockets.
    *   **Solution 1 (WebSocket)**: For `vmess+ws`, `vless+ws`, `trojan+ws`, the WASM module uses the browser's native WebSocket API to perform a real handshake and connectivity test.
    *   **Solution 2 (HTTP/Relay)**: For raw TCP protocols, we use standard HTTP latency checks where CORS permits.

## Memory Management Strategy

Running on a 7GB RAM shared runner requires strict discipline.

1.  **Streaming, Not Loading**: We never load the full dataset into a list. We process in chunks of 50.
2.  **Generators**: We use Python generators (`yield`) to pass data between stages.
3.  **Garbage Collection**: We explicitly delete large objects and call `gc.collect()` after heavy batch processing phases.
4.  **Artifact Passing**: For the "Merge" job, we don't pass raw objects. We pass optimized SQLite files and compressed JSON, minimizing the I/O overhead between GitHub Actions jobs.

## Data Flow & Sharding

To scale indefinitely, we use the **Matrix Strategy**:

1.  **Sharding**: Source files are split into `sources/batch_1.txt` through `batch_6.txt`.
2.  **Parallel Execution**: GitHub starts 6 independent VMs.
    *   VM 1 processes Batch 1.
    *   VM 2 processes Batch 2.
    *   ...
3.  **Intelligence Synchronization**:
    *   How do VMs share "History" or "Blocklist" data?
    *   They download a *common* cache at the start.
    *   At the end, they upload their *deltas* (new findings) as artifacts.
4.  **Merge Job**:
    *   The final job downloads all 6 artifact sets.
    *   It executes `scripts/merge_batches.py` to consolidate the SQLite databases and proxy lists into a single master dataset.
    *   This master dataset generates the final `metadata.json` and subscriptions.

This architecture allows ConfigStream to scale linearly. To double capacity, we just add more batch files and increase the matrix size in `pipeline.yml`.




---



# ConfigStream v2.0 Architecture

 ConfigStream v2.0 introduces advanced features for censorship resilience, decentralized infrastructure, and improved security.

 ## 1. Steganographic Delivery ("The Gallery")
 - **Objective:** Bypass DPI by disguising configs as images.
 - **Implementation:** Polyglot PNG+Zip files.
 - **Usage:** Clients download `gallery.png`, which renders as a normal image but contains an encrypted Zip payload.

 ## 2. IPFS Dead Man's Switch
 - **Objective:** Censorship-resistant fallback.
 - **Implementation:** Daily snapshots pinned to IPFS/IPNS.
 - **Failover:** If `github.io` is blocked, the client switches to IPFS gateways.
 - **Requirement:** The `publish_ipfs.py` script requires a local `ipfs` node daemon running to publish IPNS updates, or a pinning service with API support.

 ## 3. "Bring Your Own Worker" (BYOW)
 - **Objective:** Decentralize the exit node infrastructure.
 - **Mechanism:** Users deploy a Cloudflare Worker (VLESS-over-WS) and link it in the dashboard.
 - **Features:** Supports custom UUID input for authenticated workers.
 - **Benefit:** Clean IP reputation, zero cost for the platform.

 ## 4. Client-Side WASM Verification
 - **Objective:** "Residency-Based" testing.
 - **Mechanism:** A Go-based WASM module runs in the browser to test WebSocket proxies from the user's location.

 ## 5. Signed Subscription Integrity
 - **Objective:** Prevent MitM attacks.
 - **Implementation:** Ed25519 signatures attached to subscription files.
 - **Verification:** Client verifies signature against a hardcoded public key before loading.

 ## 6. Traffic Shapeshifting
 - **Objective:** Optimize multi-hop chains.
 - **Logic:** Geodesic distance calculation to ensure `Origin -> Relay -> Exit` is efficient.




---



# 03. Protocols & Parsing

ConfigStream supports a vast array of censorship-circumvention protocols. This document details the parsing logic, validation rules, and client compatibility quirks for each.

## Protocol Support Matrix

| Protocol | Parsing Module | Supported Transports | Notes |
| :--- | :--- | :--- | :--- |
| **Shadowsocks** | `parsers.shadowsocks` | TCP, UDP, Obfs | The standard. Supports modern AEAD ciphers. |
| **VMess** | `parsers.vmess` | TCP, WS, gRPC, H2 | The V2Ray workhorse. Require UUID + AlterID(0). |
| **VLESS** | `parsers.vless` | TCP, WS, gRPC, Reality | Lightweight, unencrypted (TLS-native). |
| **Trojan** | `parsers.trojan` | TCP (TLS) | Mimics HTTPS traffic. |
| **Hysteria 2** | `parsers.others` | UDP | High-performance QUIC based. |
| **Tuic** | `parsers.others` | UDP | QUIC based. |
| **WireGuard** | `parsers.others` | UDP | Supported via Cloudflare WARP integration. |
| **SSH** | `parsers.others` | TCP | Legacy tunneling. |

## Parsing Logic Diagrams

### The Parsing Pipeline

```mermaid
graph TD
    A[Raw Line] --> B{Auto-Detect Scheme}
    B -->|ss://| C(Shadowsocks Parser)
    B -->|vmess://| D(VMess Parser)
    B -->|vless://| E(VLESS Parser)

    subgraph "VMess Parsing"
        D --> D1[Decode Base64]
        D1 --> D2[Parse JSON]
        D2 --> D3[Normalize Fields]
    end

    subgraph "VLESS/Trojan Parsing"
        E --> E1[Parse URI]
        E1 --> E2[Extract Query Params]
        E2 --> E3[Extract Hashtag as Remark]
    end

    C --> F{Validation}
    D3 --> F
    E3 --> F

    F -->|Valid| G[Proxy Object]
    F -->|Invalid| H[Discard]
```

## Detailed Protocol Analysis

### 1. Shadowsocks (SS)

**Schemes**: `ss://`
**Format Variants**:
1.  **Legacy**: `ss://BASE64(method:password@host:port)`
2.  **SIP002 (Preferred)**: `ss://BASE64(method:password)@host:port`
3.  **Plain**: `ss://method:password@host:port`

**Parsing Quirks**:
*   **Padding**: Base64 strings in `ss://` often lack correct padding (`=`). Our parser automatically appends padding before decoding.
*   **Plugins**: We handle SIP003 plugins (`v2ray-plugin`, `obfs-local`). However, complex arguments are often normalized to ensure cross-client compatibility.

### 2. VMess (V2Ray)

**Schemes**: `vmess://`
**Structure**: Almost exclusively a Base64-encoded JSON object.

**Critical Fields**:
*   `id` (UUID): The user ID. Must be a valid UUID.
*   `aid` (AlterID): **Must be 0**. Non-zero AlterID is deprecated and insecure. We force-set this to 0.
*   `net` (Network): Can be `tcp`, `ws`, `grpc`, `h2`.
*   `type` (Header): For TCP, can be `http` (obfuscation).
*   `scy` (Security): Usually `auto`.

**Transport Specifics**:
*   **WebSocket (WS)**: Requires `path` and `host` (for the Host header).
*   **gRPC**: Requires `serviceName`.
*   **H2**: Requires `path`.

### 3. VLESS (V2Ray / Xray)

**Schemes**: `vless://`
**Structure**: `vless://UUID@HOST:PORT?params#Remark`

**VLESS Reality (The Game Changer)**:
Reality replaces standard TLS with a "steal" mechanism.
*   **Required Fields**:
    *   `pbk` (Public Key): The public key of the Reality server.
    *   `sid` (Short ID): Hex string.
    *   `sni`: The domain being mimicked (e.g., `www.microsoft.com`).
    *   `fp`: Browser fingerprint (e.g., `chrome`).

**Validation Rule**: If `security=reality`, we check for `pbk` and `sid`. If missing, the proxy is invalid.

### 4. Trojan

**Schemes**: `trojan://`
**Structure**: `trojan://PASSWORD@HOST:PORT?params#Remark`

**Mechanism**:
Trojan listens on port 443. It performs a real TLS handshake. If the first packet after handshake doesn't contain the correct password hash, it proxies the traffic to a fallback web server (like Nginx). To a censor, it looks exactly like browsing a website.

### 5. Hysteria 2 & Tuic

**Schemes**: `hysteria2://`, `hy2://`, `tuic://`

**Characteristics**:
*   **UDP-based**: Uses QUIC.
*   **Congestion Control**: Designed to bully through packet loss.
*   **Obfuscation**: Uses a password to encrypt headers.

**Client Support**:
*   Sing-box: Native support.
*   Clash Meta: Native support.
*   Clash Premium: No support.
*   Surge: Partial support.

## Deduplication Logic

We define a "Unique Proxy" not by the full link string, but by its connectivity endpoint and credentials.

**Composite Key Construction**:


In [ ]:
def proxy_unique_key(p: Proxy) -> tuple:
    # Scheme + Host + Port + Username/UUID + Path
    return (
        p.protocol,
        p.address.lower(),
        p.port,
        p.username or p.uuid or "none",
        p.details.get("path", "")
    )



This prevents duplicate entries where the only difference is the "Remark" or the "SNI" (if SNI is not critical for identity).

## Validation Rules

Before a proxy enters the testing phase, it must pass the **Gatekeeper**:

1.  **Port Range**: 1-65535.
2.  **Host Validity**:
    *   Must not be a private IP (`192.168.x.x`, `10.x.x.x`, `127.0.0.1`) unless `allow_private` is set (dev mode).
    *   Must not be a broadcast or multicast address.
3.  **Field Integrity**:
    *   VMess: Must have `id` (UUID).
    *   Shadowsocks: Must have `method` and `password`.
    *   Trojan: Must have `password`.
4.  **Scheme Enforcement**: We reject "naked" links (IP:Port) unless they are in a specific raw format file.




---



# 07. Security & Privacy

ConfigStream operates in a hostile environment. We deal with circumvention tools, which attracts attention from censors and malicious actors. Security is not an afterthought; it is the foundation.

## The Threat Model

We defend against three primary threats:
1.  **Pollution Attacks**: Malicious actors flooding the repo with fake or blocked proxies to dilute the pool.
2.  **Honeypots**: State actors deploying logging nodes to track users.
3.  **Takedowns**: Hosting providers removing the repo due to abuse reports.

## Mitigation Strategies

### 1. Pollution Defense (Anomaly Detection)
We use statistical analysis to detect pollution.
*   **Subnet Analysis**: Spammers rarely have diverse IPs. They spin up 100 containers on one Vultr/DigitalOcean droplet. If we see >90% of a batch coming from one /24 subnet, we drop it.
*   **Sequential Ports**: Proxies on `8001, 8002, 8003` are almost always a port scan or a single multi-user node. We deduplicate these aggressively.

### 2. Honeypot Detection
State actors often deploy "fake" open proxies to log traffic.

Current implementation focusses on **passive** intelligence only:

*   **VirusTotal Integration**: Our honeypot guard (`src/configstream/security/honeypot.py`) uses `check_ip_reputation()` to check whether an IP has been flagged as malicious.
    *   If VirusTotal reports `malicious > 0`, the proxy is flagged and removed from the pool.
    *   If the `VT_API_KEY` is missing, we **log a warning** and fail open (no blocking), so operators are aware that honeypot reputation checks are effectively disabled.
*   **No Active Scanning**: Behavioral checks like open resolvers, echo servers, and header fingerprinting are **documented but not enabled** in production to respect a strict “Zero Abuse” policy and avoid port‑scan behaviour.
    *   The `check_common_honeypot_ports()` and `check_traffic_interception()` helpers are intentionally stubs.
    *   Future work may add passive heuristics here (e.g. offline logs analysis) without probing the remote hosts.

### 3. Proxy Washing (IP Reputation)
Many "free" proxies are on IPs that are flagged by Cloudflare, Google, or other major providers. They connect, but they get CAPTCHAs or 403s.
*   **Concept**: We use the `ProxyWasher` (`src/configstream/intelligence/washer.py`).
*   **Mechanism**:
    *   `User -> [Dirty Proxy] -> [WARP Interface] -> Target`
*   **Implementation**:
    *   We generate a complex **Sing-box** configuration chain.
    *   The "Dirty Proxy" is the `outbound`.
    *   The `WARP` WireGuard tunnel is a second `outbound`.
    *   A `route` rule sends traffic from the Proxy outbound *into* the WARP outbound.
    *   The user gets a clean Cloudflare IP.
*   **Deterministic Key Assignment**: We map the Proxy ID to a specific WARP key using a hash (now based on SHA‑256) so that the same proxy always gets the same "Identity" (WARP key), preventing session churn.
*   **Candidate Selection**: When a WARP key pool is configured, we wash **all working proxies** (not only those tagged `dirty_ip`/`insecure`) to ensure that untagged but risky nodes still benefit from a clean egress IP.

### 4. TLS Fingerprinting (uTLS)
Standard Python `requests` or `ssl` libraries have a very distinct TLS fingerprint (JA3). Firewalls block this immediately.
*   **Solution**: We use **uTLS** (in our Go sidecar).
*   **Randomization**: We randomize the Client Hello packet to mimic:
    *   Chrome 120
    *   Firefox 118
    *   Safari 17
*   **Result**: We can successfully test and connect to proxies that block non-browser traffic.

### 5. Intranet Bridge
Some proxies are located inside restrictive domestic networks (e.g., Iran, China) and cannot reach the global internet directly, but *can* reach other domestic servers.
*   **Mechanism**: We chain these "Intranet" proxies through a "Bridge" proxy (a domestic server with international access, or a relay).
*   **Routing**: We create specific routing rules in `singbox.json` to tunnel traffic intelligently.

## Secrets Management

*   **No API Keys in Code**: We use GitHub Secrets.
*   **Gitleaks**: We run `gitleaks` in CI to catch accidental commits.
*   **Sanitization**: The pipeline strips `user`, `pass`, and `uuid` from logs. We only log the `hash(proxy)` for debugging.

## VirusTotal Integration

We cross-reference proxy IPs with VirusTotal's database.
*   **Metric**: "Malicious Votes".
*   **Threshold**: If > 3 vendors flag an IP as malware/botnet, we drop it.
*   **Caching**: We cache VT results for 7 days to respect API limits.
*   **Failure Mode**: If the API key is not configured or the API fails, we log a **warning** and fail open for that check only (the proxy may still be rejected by other validators such as blocklists or DNS rules).

## Anomaly Detection & Fail‑Open Policy

The `AnomalyDetector` (`src/configstream/anomaly.py`) protects against **pollution attacks** by modelling per‑source history and identifying massive spikes or drops.

*   For established sources, Isolation Forest / Z‑score heuristics are used to detect outliers in batch size.
*   For new or small sources, simple heuristics guard against “sudden massive yield”.
*   **Failure Mode**: If the anomaly database is temporarily unavailable (e.g. SQLite error), we now **fail open**:
    *   The source is **allowed** for this run.
    *   An error is logged with `DB Error (Fail Open)` so operators can fix the underlying storage problem.
    *   This prevents a transient DB issue from blocking **all** upstreams and silently collapsing the pipeline.


